# NewsAPI

## Set Up

In [1]:
import os
from dotenv import load_dotenv, dotenv_values
from pathlib import Path

In [2]:
load_dotenv(Path("/Users/tanaphan/Documents/STUDY/UoBristol/Final_Project/team_21_lloyds/.env"))

True

In [3]:
API_KEY_NEWS = os.getenv("API_KEY_NEWS")

print(API_KEY_NEWS)

5f1946f04b7642ab944af4806cdc7e4b


## EDA

In [4]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#https://newsapi.org/v2/everything
#https://newsapi.org/v2/top-headlines

# FUNCTION: Pull NewsAPI - Everything Endpoint
def pull_news(KEYWORD):
    response = requests.get("https://newsapi.org/v2/everything", params={
        "q": f'"{KEYWORD}"', # Search using this Keyword
        "language": "en",
        "sortBy": "publishedAt", # relevancy, popularity, publishedAt
        "pageSize": 100, # 100 Max
        "page": 1,
        "apiKey": API_KEY_NEWS,
    })
    data = response.json()
    articles = data.get("articles", [])
    print(f"API status     : {data.get('status')}")
    print(f"Total results: {data.get('totalResults')}")
    print(f"Total articles pulled: {len(articles)}")
    if data.get("status") == "error":
        print(f"Error code   : {data.get('code')}")
        print(f"Error message: {data.get('message')}")
    df = pd.json_normalize(articles)
    return df

In [5]:
# Set Keyword
KEYWORD = "lidl"

print("="*15)
print(f"News for {KEYWORD}")
print("="*15)

df = pull_news(KEYWORD)

# Shape
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns and types:")
print(df.dtypes)

News for lidl
API status     : ok
Total results: 174
Total articles pulled: 76

Shape: 76 rows x 9 columns

Columns and types:
author         object
title          object
description    object
url            object
urlToImage     object
publishedAt    object
content        object
source.id      object
source.name    object
dtype: object


In [6]:
print(df.head)

<bound method NDFrame.head of               author                                              title  \
0       Caitlin Leng  Urgent recall on apples and kiwi fruit sold at...   
1         Aengus Cox  Late payments among issues for suppliers, surv...   
2       Michael Kwan  Fascinating kasszinno Tactics That Can Help Yo...   
3       Michael Kwan      Here Is What You Should Do For Your red tiger   
4        Andrew Levy  Farm which supplied pork to major supermarkets...   
..               ...                                                ...   
71        Mike Peter  Balsamo wins fourth consecutive Giro sprint stage   
72  Chrissy Callahan  15 National Donut Day Deals: Get Free Dunkin’,...   
73       Gabe Hauari  National Donut Day 2026 deals, freebies at Kri...   
74  Henry Winchester             Madison Flux Mens EIT Padded Bib Short   
75       Martin Wall  Lidl warns planning system is being exploited ...   

                                          description  \
0   An alert

In [7]:
# Missing Values
print("Missing Value:")
print(df.isnull().sum())

# Duplication
print("\nDuplicatation:")
dup_title = df.duplicated(subset=["title"]).sum()
dup_description = df.duplicated(subset=["description"]).sum()
dup_url   = df.duplicated(subset=["url"]).sum()

print(f"Duplicate titles : {dup_title}")
print(f"Duplicate description : {dup_description}")
print(f"Duplicate URLs   : {dup_url}")

Missing Value:
author          6
title           0
description     0
url             0
urlToImage      3
publishedAt     0
content         0
source.id      55
source.name     0
dtype: int64

Duplicatation:
Duplicate titles : 1
Duplicate description : 0
Duplicate URLs   : 0


In [8]:
# Articles overtime
print("Article Timeline")
df["publishedAt"] = pd.to_datetime(df["publishedAt"])
df["date"] = df["publishedAt"].dt.date # create a column "date"
daily_counts = df.groupby("date").size().reset_index(name="count")
print(daily_counts)
plt.figure(figsize=(len(daily_counts) * 0.7, 4))
plt.plot(daily_counts["date"], daily_counts["count"], marker="o", color="blue")
plt.fill_between(daily_counts["date"], daily_counts["count"], alpha=0.1, color="blue")
plt.title(f"Articles per day | Keyword = {KEYWORD}")
plt.xlabel("Date")
plt.ylabel("Article Count")
plt.savefig(f"newsapi_{KEYWORD}_timeline.png") # Save graph
plt.close()

# Source Distribution
print("\nSource Distribution")
source_counts = df.groupby("source.name").size().reset_index(name="count").sort_values("count", ascending=False)
print(source_counts)
plt.figure(figsize=(10, len(source_counts) * 0.4))
plt.barh(source_counts["source.name"], source_counts["count"], color="green")
plt.title(f"Articles by source | Keyword = {KEYWORD}")
plt.xlabel("Article count")
plt.savefig(f"newsapi_{KEYWORD}_sources.png", bbox_inches="tight")
plt.close()

# Text Length
df["title_len"] = df["title"].fillna("").apply(len)
df["description_len"] = df["description"].fillna("").apply(len)

# Save as .CSV file
output_cols = ["date", "source.name", "title", "description", "content",
               "title_len", "description_len"]
df[output_cols].to_csv(f"newsapi_{KEYWORD}.csv", index=False)

Article Timeline
          date  count
0   2026-06-04      5
1   2026-06-05     13
2   2026-06-06      4
3   2026-06-08      1
4   2026-06-09      3
5   2026-06-10      5
6   2026-06-11      2
7   2026-06-12      5
8   2026-06-13      1
9   2026-06-14      3
10  2026-06-15      3
11  2026-06-16      3
12  2026-06-17      8
13  2026-06-18      2
14  2026-06-19      7
15  2026-06-20      2
16  2026-06-21      3
17  2026-06-23      5
18  2026-06-24      1

Source Distribution
                   source.name  count
22             The Irish Times      9
5                Dailymail.com      9
10                   Inrng.com      8
15                         RTE      8
1                     BBC News      7
25               TheJournal.ie      4
9               Ibtimes.com.au      3
14               Protothema.gr      2
11                Johnchow.com      2
31                     road.cc      2
20                   TechRadar      1
26                   USA Today      1
24                       The